# 1 · SFT (Supervised Fine-Tuning)
**Idea:** show the model examples with the correct answer. It learns to answer the same way.

`black sneakers under 50 dollars` → `FIND shoes color=BK max=50`

The model cannot guess this: the format and the color codes (`BK` = black) are the shop's own rules.

## Step 1 · Load a small model

In [2]:
import warnings, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings('ignore')            # hide technical warnings

MODEL = 'Qwen/Qwen3.5-0.8B'                  # small open model (0.8 billion parameters)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16, device_map='cuda')

Loading weights: 100%|██████████| 320/320 [00:01<00:00, 317.37it/s]


## Step 2 · Write the examples
25 examples to learn from, and 5 to test. The test examples are never used for training.

In [4]:
TRAIN = [  # (request, correct command)
    ('red tee',                           'FIND shirt color=RD'),
    ('blue shirt',                        'FIND shirt color=BL'),
    ('black t-shirt under 20 dollars',    'FIND shirt color=BK max=20'),
    ('white tee below 30',                'FIND shirt color=WH max=30'),
    ('I want a red t-shirt',              'FIND shirt color=RD'),
    ('white shirt',                       'FIND shirt color=WH'),
    ('black sneakers',                    'FIND shoes color=BK'),
    ('white trainers under 80 dollars',   'FIND shoes color=WH max=80'),
    ('blue shoes',                        'FIND shoes color=BL'),
    ('red sneakers below 50',             'FIND shoes color=RD max=50'),
    ('show me black trainers',            'FIND shoes color=BK'),
    ('red shoes under 30 dollars',        'FIND shoes color=RD max=30'),
    ('blue trainers',                     'FIND shoes color=BL'),
    ('black coat',                        'FIND jacket color=BK'),
    ('blue jacket under 80 dollars',      'FIND jacket color=BL max=80'),
    ('white coat below 50',               'FIND jacket color=WH max=50'),
    ('I want a red jacket',               'FIND jacket color=RD'),
    ('blue coat',                         'FIND jacket color=BL'),
    ('black jacket below 80',             'FIND jacket color=BK max=80'),
    ('red hoodie',                        'FIND hoodie color=RD'),
    ('white sweatshirt under 30 dollars', 'FIND hoodie color=WH max=30'),
    ('black hoodie below 20',             'FIND hoodie color=BK max=20'),
    ('show me a blue sweatshirt',         'FIND hoodie color=BL'),
    ('white hoodie',                      'FIND hoodie color=WH'),
    ('red sweatshirt below 50',           'FIND hoodie color=RD max=50'),
]

TEST = [
    ('blue tee under 50 dollars',         'FIND shirt color=BL max=50'),
    ('white sneakers',                    'FIND shoes color=WH'),
    ('red coat below 30',                 'FIND jacket color=RD max=30'),
    ('black sweatshirt',                  'FIND hoodie color=BK'),
    ('show me a white t-shirt',           'FIND shirt color=WH'),
]

def question(request):
    return [{'role': 'user', 'content': f'Search command for: {request}'}]

## Step 3 · Ask the model before training
The model has never seen these rules, so it should get most answers wrong.

In [9]:
def ask(model, request):
    inputs = tokenizer.apply_chat_template(question(request), add_generation_prompt=True,return_tensors='pt', return_dict=True).to('cuda')
    output = model.generate(**inputs, max_new_tokens=50, do_sample=False)
    answer = output[0][inputs['input_ids'].shape[1]:]  # keep only the new words
    return tokenizer.decode(answer, skip_special_tokens=True).strip()

def test(model):
    model.eval() # answering mode
    right = 0
    for request, command in TEST:
        answer = ask(model, request)
        if answer == command:
            right += 1
            print('right:', request, '->', answer)
        else:
            print('wrong:', request, '->', answer, '   expected:', command)
    print(right, 'of', len(TEST), 'right')

test(model)

right: blue tee under 50 dollars -> FIND shirt color=BL max=50
right: white sneakers -> FIND shoes color=WH
right: red coat below 30 -> FIND jacket color=RD max=30
right: black sweatshirt -> FIND hoodie color=BK
right: show me a white t-shirt -> FIND shirt color=WH
5 of 5 right


## Step 4 · Train
Each example is a request (the prompt) and its correct command (the completion). The model learns to write the completion.

- **LoRA:** only a small add-on (about 1% of the model) is trained.
- **Epochs:** how many times each example is seen.
- **Loss:** how wrong the model is. Watch it go down.

In [11]:
from datasets import Dataset
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

data = Dataset.from_list([
    {'prompt': question(request), 'completion': [{'role': 'assistant', 'content': command}]}
    for request, command in TRAIN
])

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=data,
    peft_config=LoraConfig(r=16, lora_alpha=32, target_modules='all-linear'),   # the small add-on
    args=SFTConfig(
        num_train_epochs=10,     # each example is seen 10 times
        learning_rate=2e-4,   # size of each small adjustment
        logging_steps=1,     # print the loss every 4 steps
        save_strategy='no',   # do not save copies to disk
    ),
)
trainer.train()

Truncating train dataset: 100%|██████████| 25/25 [00:00<00:00, 5134.54 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 25/25 [00:00<00:00, 6397.27 examples/s]


Step,Training Loss
1,6.431316
2,3.210324
3,1.867778
4,0.667670
5,0.327294
6,0.140566
7,0.116095
8,0.020300
9,0.314950
10,0.078010


TrainOutput(global_step=40, training_loss=0.3407802658402943, metrics={'train_runtime': 19.123, 'train_samples_per_second': 13.073, 'train_steps_per_second': 2.092, 'total_flos': 26813802096000.0, 'train_loss': 0.3407802658402943, 'epoch': 10.0})

## Step 5 · Ask the model after training
The same 5 test requests. The model never saw them, so right answers mean it learned the rules.

In [12]:
test(trainer.model)

right: blue tee under 50 dollars -> FIND shirt color=BL max=50
right: white sneakers -> FIND shoes color=WH
right: red coat below 30 -> FIND jacket color=RD max=30
right: black sweatshirt -> FIND hoodie color=BK
right: show me a white t-shirt -> FIND shirt color=WH
5 of 5 right
